# 🧲 What Are Embeddings?

**Day 5 — Notebook 1 of 4 | Estimated Time: 25 minutes**

---

## What You'll Learn
- What embeddings are and why they are fundamental to modern AI
- How text is converted into vectors of numbers
- What dimensionality means and why it matters
- The intuition behind "meaning as position in space"

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" numpy matplotlib scikit-learn

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types

from google.genai import errors

try:
    client = genai.Client(api_key=get_api_key())
except errors.APIError as e:
    print(f"Gemini API Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
else:
    MODEL_ID = "gemini-2.5-flash"
    print("✅ Ready!")
EMBEDDING_MODEL = "gemini-embedding-001"

---

## 1. The Core Idea

An **embedding** is a way of representing text (or images, audio, etc.) as a **list of numbers** — a vector — such that **semantically similar things are close together** in that numerical space.

```
"king"   → [0.21, -0.45, 0.83, 0.12, ...]   (768 numbers)
"queen"  → [0.19, -0.41, 0.79, 0.15, ...]   ← very close to "king"
"banana" → [-0.55, 0.72, -0.31, 0.88, ...]  ← very far from "king"
```

This is the magic: **meaning becomes geometry**. Similarity in meaning = closeness in space.

### Why do we need embeddings?

Computers can't work with raw text directly. They need numbers. But naïve approaches (like one-hot encoding) don't capture meaning. Embeddings solve this — they give each piece of text a position in a high-dimensional space where position encodes meaning.

---

## 2. Your First Embedding

In [ ]:
# Generate an embedding for a single sentence
result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents="The weather is beautiful today.",
)

embedding = result.embeddings[0].values
print(f"Type: {type(embedding)}")
print(f"Number of dimensions: {len(embedding)}")
print(f"First 10 values: {[round(v, 4) for v in embedding[:10]]}")
print(f"Min value: {min(embedding):.4f}")
print(f"Max value: {max(embedding):.4f}")

Every piece of text maps to a **3072-dimensional vector** with `gemini-embedding-001`. That's 768 numbers, each capturing a different aspect of the meaning.

---

## 3. Similar Texts → Similar Vectors

Let's compare embeddings for similar and dissimilar texts. We'll use **cosine similarity** as our distance metric (we'll explain this fully in Notebook 3 — for now just watch the numbers).

In [ ]:
def get_embedding(text: str) -> np.ndarray:
    """Get the embedding vector for a piece of text."""
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
    )
    return np.array(result.embeddings[0].values)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Calculate how similar two vectors are (1.0 = identical, 0.0 = unrelated)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Anchor sentence
anchor = "I love eating pizza for dinner."
anchor_vec = get_embedding(anchor)

comparisons = [
    "I enjoy having pasta in the evening.",      # Similar (food + dinner)
    "Pizza is my favourite meal.",                # Very similar
    "The stock market closed up 2% today.",       # Unrelated
    "She plays chess every weekend.",             # Unrelated
    "We ordered takeaway food last night.",       # Somewhat related
]

print(f"Anchor: '{anchor}'\n")
for text in comparisons:
    vec = get_embedding(text)
    sim = cosine_similarity(anchor_vec, vec)
    bar = "█" * int(sim * 30)
    print(f"  {sim:.3f} {bar}")
    print(f"         '{text}'")
    print()

---

## 4. Semantic vs. Lexical Similarity

Embeddings capture **meaning**, not just words. Two sentences can use completely different words but have the same meaning — and their embeddings will be close.

In [ ]:
pairs = [
    # High semantic similarity, different words
    ("The car broke down on the highway.", "The vehicle stopped working on the motorway."),
    # Low semantic similarity, similar words
    ("The bank was closed on Sunday.", "She sat by the river bank watching ducks."),
    # Opposite meaning
    ("I love this product.", "I hate this product."),
    # Identical meaning, different languages concept
    ("Hello, how are you?", "Hi, how are you doing?"),
]

print("Semantic Similarity Experiments:\n")
for a, b in pairs:
    vec_a = get_embedding(a)
    vec_b = get_embedding(b)
    sim = cosine_similarity(vec_a, vec_b)
    print(f"  A: '{a}'")
    print(f"  B: '{b}'")
    print(f"  Similarity: {sim:.3f}\n")

**Key observations:**
- Same meaning, different words → high similarity
- Polysemy ("bank") → embeddings may reflect the dominant meaning
- Opposites can still have moderately high similarity (they're in the same topic space)

---

## 5. Visualising Embeddings in 2D

3072 dimensions is impossible to visualise directly. We can use **PCA** (Principal Component Analysis) to compress the embeddings down to 2D for visualisation. This loses some information, but gives us intuition.

In [ ]:
# A set of texts across different semantic clusters
texts = [
    # Technology
    "Machine learning is a type of artificial intelligence.",
    "Neural networks are inspired by the human brain.",
    "Python is a popular programming language for data science.",
    # Food
    "Pasta is a traditional Italian dish.",
    "Sushi originated in Japan.",
    "Pizza is loved worldwide.",
    # Sports
    "Football is the most popular sport globally.",
    "The Olympics are held every four years.",
    "Tennis requires agility and precision.",
    # Finance
    "Stocks represent ownership in a company.",
    "Interest rates affect borrowing costs.",
    "Inflation reduces purchasing power.",
]

labels = ["Tech"] * 3 + ["Food"] * 3 + ["Sports"] * 3 + ["Finance"] * 3
colors = {"Tech": "#4285f4", "Food": "#ea4335", "Sports": "#34a853", "Finance": "#fbbc04"}

# Get embeddings for all texts
print("Generating embeddings...")
vectors = np.array([get_embedding(t) for t in texts])

# Reduce to 2D with PCA
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(vectors)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))
for i, (x, y) in enumerate(coords_2d):
    label = labels[i]
    color = colors[label]
    ax.scatter(x, y, c=color, s=120, zorder=3)
    ax.annotate(
        texts[i][:35] + "...",
        (x, y),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=8,
        color=color,
    )

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in colors.items()]
ax.legend(handles=legend_elements, loc="upper right")

ax.set_title("Embedding Clusters (PCA 2D projection)")
ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nVariance explained by 2 PCA components: {pca.explained_variance_ratio_.sum():.1%}")

**What to observe:** Texts about the same topic should cluster together in the 2D projection, even though the embedding model was never told about these categories.

---

## 6. Embedding Dimensions: What Do They Encode?

Each of the 3072 dimensions encodes a **latent feature** of the text. The model learned these features from massive amounts of training data. We can't directly interpret what each dimension means, but we can observe patterns.

In [ ]:
# Compare embedding norms and distributions
sample_texts = [
    "cat",
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming every industry at an unprecedented pace.",
]

print("Embedding statistics:\n")
for text in sample_texts:
    vec = get_embedding(text)
    print(f"Text: '{text[:60]}'")
    print(f"  Dimensions: {len(vec)}")
    print(f"  L2 norm: {np.linalg.norm(vec):.4f}")
    print(f"  Mean: {np.mean(vec):.4f}, Std: {np.std(vec):.4f}")
    print()

---

## 🏋️ Exercise 1: Find the Odd One Out

Given a list of texts, use embeddings to find the one that is most semantically different from the others.

In [ ]:
# Exercise 1: Semantic odd-one-out
# TODO: Find which text in this list is most different from the others
# Hint: Calculate the average similarity of each text to all others. The lowest = odd one out.

texts = [
    "The Eiffel Tower is in Paris.",
    "The Colosseum is a famous Roman landmark.",
    "Photosynthesis converts sunlight into energy.",  # Odd one out?
    "Big Ben is a clock tower in London.",
    "The Statue of Liberty stands in New York harbour.",
]

# TODO: Implement this
# 1. Get embeddings for all texts
# 2. For each text, compute its average cosine similarity to all other texts
# 3. Print the text with the lowest average similarity

# Your code here

---

## 🏋️ Exercise 2: Semantic Grouping

You have a mixed list of sentences. Without using labels, group them by topic using their embeddings.

In [ ]:
# Exercise 2: Group by semantic similarity
mixed_texts = [
    "Insulin regulates blood sugar levels.",
    "The Python pandas library is used for data manipulation.",
    "Antibiotics treat bacterial infections.",
    "SQL is used to query relational databases.",
    "The heart pumps blood through the circulatory system.",
    "Docker containers package applications and their dependencies.",
    "Vaccines stimulate the immune system.",
    "APIs allow software systems to communicate.",
]

# TODO: 
# 1. Get embeddings for all texts
# 2. Use the cosine_similarity function to build a similarity matrix
# 3. Print the top 2 most similar pairs of sentences
# Bonus: Try using sklearn's KMeans with n_clusters=2 to automatically group them

# Your code here

---

## Key Takeaways

| Concept | Summary |
|---------|----------|
| **Embedding** | A fixed-length vector representation of text |
| **Dimensionality** | Number of values in the vector (e.g., 3072 for `gemini-embedding-001`) |
| **Semantic similarity** | Texts with similar meanings have vectors that point in similar directions |
| **Cosine similarity** | Measures the angle between two vectors — 1.0 = same direction, 0.0 = orthogonal |
| **Visualisation** | PCA reduces high-dimensional embeddings to 2D for inspection |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| The Illustrated Word2Vec | Blog | [jalammar.github.io/illustrated-word2vec](https://jalammar.github.io/illustrated-word2vec/) |
| Gemini Embeddings API | Docs | [ai.google.dev/gemini-api/docs/embeddings](https://ai.google.dev/gemini-api/docs/embeddings) |
| Word Embeddings — StatQuest | Video | [youtube.com/watch?v=viZrOnJclY0](https://www.youtube.com/watch?v=viZrOnJclY0) |

---

**Next up:** [02_Generating_Embeddings_with_Gemini.ipynb](./02_Generating_Embeddings_with_Gemini.ipynb)